In [ ]:
import sys, os, shutil, tempfile, warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._ts_dataloader      import DataLoaderFactory
from dataloaders.ts_sharding        import (
    write_sharded_dataset,
    ShardedTrainDataset,
    ShardedValDataset,
    ShardedTestDataset,
)
from common.train import train, eval_test
from models.linear import TinyLinearModel
from losses.torch_losses import get_loss
from common.ensembling import Ensembler

warnings.filterwarnings("ignore")
print("imports OK")


In [ ]:
print("=" * 60)
print("TEST — single dataset, forecast ensembling")
print("=" * 60)

T, H, C = 10, 5, 2

# rolling pattern: window t predicts [(t%5)+1, ...]
pattern = np.array([1, 2, 3, 4, 5], dtype=float)
preds_single = np.stack([np.roll(pattern, -t) for t in range(T)])  # (T, H)

# same for both channels, add batch dim
preds = np.stack([preds_single, preds_single], axis=-1)[None]      # (1, T, H, C)
print(preds.shape)  # (1, 10, 5, 2)
print(preds[0, :, :, 0])

ensembler = Ensembler(ensemble_method='mean')
ensembled_windows = ensembler.ensemble(preds)

assert ensembled_windows.shape == (1, 10, 5, 2), "Ensembled window shape is incorrect. It should be (1, 10, 5, 2)."
assert np.allclose(a=ensembled_windows, b=preds, atol=1e-9), "Ensembled windows should match predictions in this test."

print("\n✓ TEST PASSED")